# Word Embeddings dengan TensorFlow/Keras

Notebook ini adalah versi pembelajaran berbahasa Indonesia yang mengikuti alur tutorial **Word Embeddings** dari TensorFlow. Tutorial aslinya dapat diakses [di sini](https://www.tensorflow.org/tutorials/text/word_embeddings).

## 1. Import Library

Kita mulai dengan mengimpor library yang dibutuhkan.

Penjelasan singkat:

- `tensorflow`: library utama untuk membuat dan melatih model neural network.
- `Sequential`: cara sederhana membuat model berlapis secara berurutan.
- `TextVectorization`: layer untuk mengubah teks mentah menjadi angka.
- `Embedding`: layer untuk mengubah indeks kata menjadi vektor padat.
- `GlobalAveragePooling1D`: layer sederhana untuk merangkum banyak embedding kata menjadi satu vektor dokumen.
- `Dense`: layer neural network biasa untuk klasifikasi.

In [ ]:
import io
import os
import re
import shutil
import string

import tensorflow as tf

from tensorflow.keras import Sequential
from tensorflow.keras.layers import Dense, Embedding, GlobalAveragePooling1D
from tensorflow.keras.layers import TextVectorization

print("TensorFlow version:", tf.__version__)

## 2. Mengapa Teks Harus Diubah Menjadi Angka?

Model machine learning tidak dapat langsung memahami teks seperti manusia. Model menerima input berupa angka, biasanya dalam bentuk vektor atau tensor.

Contoh kalimat:

> "film ini bagus"

Bagi manusia, kalimat tersebut mudah dipahami. Namun bagi model, kata `"film"`, `"ini"`, dan `"bagus"` harus diubah menjadi angka terlebih dahulu.

Ada beberapa cara umum untuk merepresentasikan teks:

1. **One-hot encoding**
2. **Integer encoding**
3. **Word embedding**

Kita bahas secara bertahap.

## 3. Contoh Sederhana: One-Hot Encoding

Pada one-hot encoding, setiap kata direpresentasikan sebagai vektor panjang yang hampir semuanya berisi nol, kecuali satu posisi yang bernilai satu.

Misalnya vocabulary kita adalah:

```text
["film", "ini", "bagus", "buruk"]
```

Maka kata `"film"` dapat direpresentasikan sebagai:

```text
[1, 0, 0, 0]
```

Kata `"bagus"` dapat direpresentasikan sebagai:

```text
[0, 0, 1, 0]
```

Kelemahannya:

- Vektornya bisa sangat panjang jika vocabulary besar.
- Banyak nilai nol, sehingga tidak efisien.
- Tidak menangkap hubungan makna antarkata.

In [ ]:
# Contoh sederhana one-hot encoding secara manual

vocab_kecil = ["film", "ini", "bagus", "buruk"]
word_to_index = {word: idx for idx, word in enumerate(vocab_kecil)}

def one_hot(word, vocabulary):
    vector = [0] * len(vocabulary)
    index = word_to_index[word]
    vector[index] = 1
    return vector

print("Vocabulary:", vocab_kecil)
print("One-hot untuk kata 'film' :", one_hot("film", vocab_kecil))
print("One-hot untuk kata 'bagus':", one_hot("bagus", vocab_kecil))

## 4. Integer Encoding

Cara kedua adalah memberi setiap kata sebuah nomor unik.

Contoh:

```text
film  -> 1
ini   -> 2
bagus -> 3
buruk -> 4
```

Kalimat:

```text
film ini bagus
```

dapat diubah menjadi:

```text
[1, 2, 3]
```

Cara ini lebih hemat dibanding one-hot encoding, tetapi ada masalah penting:

> Nomor kata bersifat arbitrer. Artinya, angka 3 untuk `"bagus"` tidak berarti kata tersebut benar-benar lebih besar atau lebih dekat secara makna dengan kata tertentu.

Model bisa salah menafsirkan angka tersebut sebagai urutan atau besaran.

In [ ]:
# Contoh integer encoding sederhana

kalimat = "film ini bagus"
encoded = [word_to_index[word] + 1 for word in kalimat.split()]

print("Kalimat asli :", kalimat)
print("Hasil encoding:", encoded)

## 5. Word Embedding: Ide Utama

**Word embedding** adalah representasi kata dalam bentuk vektor padat berisi bilangan desimal.

Contoh sederhana:

```text
bagus  -> [0.21, -0.54, 0.88, 0.10]
buruk  -> [-0.30, 0.76, -0.12, 0.45]
film   -> [0.11, 0.09, -0.33, 0.72]
```

Perbedaannya dengan one-hot encoding:

- One-hot dibuat secara manual berdasarkan posisi kata.
- Embedding dipelajari oleh model selama proses training.

Intuisi sederhananya:

> Jika dua kata sering muncul dalam konteks yang mirip, embedding-nya cenderung menjadi lebih dekat di ruang vektor.

Dalam Keras, layer `Embedding` dapat dipahami sebagai **lookup table**. Inputnya adalah indeks kata, outputnya adalah vektor embedding untuk indeks tersebut.

## 6. Demonstrasi Layer `Embedding`

Sebelum masuk ke dataset IMDb, kita coba dulu layer `Embedding` pada data kecil.

Misalnya:

```python
Embedding(1000, 5)
```

Artinya:

- Model menyiapkan embedding untuk maksimal 1000 token/kata.
- Setiap token direpresentasikan sebagai vektor berdimensi 5.

Pada awalnya, nilai embedding masih acak. Setelah training, nilai tersebut berubah karena dipelajari oleh model.

In [ ]:
# Embed vocabulary berukuran 1000 ke dalam vektor berdimensi 5.
embedding_layer = tf.keras.layers.Embedding(input_dim=1000, output_dim=5)

# Kita masukkan tiga indeks kata: 1, 2, dan 3.
result = embedding_layer(tf.constant([1, 2, 3]))

print("Shape output:", result.shape)
print("Isi embedding:")
print(result.numpy())

Penjelasan output:

- Input berisi 3 indeks kata: `[1, 2, 3]`.
- Setiap indeks diubah menjadi vektor berdimensi 5.
- Maka output memiliki shape `(3, 5)`.

Sekarang kita coba input berupa batch kalimat. Misalnya ada 2 kalimat, masing-masing memiliki 3 token.

In [ ]:
# Contoh batch:
# Kalimat 1: [0, 1, 3]
# Kalimat 2: [3, 4, 5]

result = embedding_layer(tf.constant([[0, 1, 3], [3, 4, 5]]))

print("Shape output:", result.shape)

Penjelasan shape:

```text
(2, 3, 5)
```

Artinya:

- `2` = jumlah kalimat dalam batch.
- `3` = jumlah token per kalimat.
- `5` = dimensi embedding untuk setiap token.

Jadi, layer `Embedding` menambahkan satu dimensi baru, yaitu dimensi vektor embedding.

## 7. Download Dataset IMDb

Sekarang kita gunakan **Large Movie Review Dataset** dari IMDb.

Dataset ini berisi ulasan film dengan dua label:

- `pos`: ulasan positif.
- `neg`: ulasan negatif.

Tujuan kita adalah melatih model untuk membedakan ulasan positif dan negatif.

In [ ]:
url = "https://ai.stanford.edu/~amaas/data/sentiment/aclImdb_v1.tar.gz"

dataset = tf.keras.utils.get_file(
    "aclImdb_v1.tar.gz",
    url,
    untar=True,
    cache_dir=".",
    cache_subdir=""
)

print("Dataset berhasil diunduh atau sudah tersedia di:")
print(dataset)

## 8. Menentukan Lokasi Folder Dataset

Struktur folder hasil ekstraksi dapat sedikit berbeda tergantung versi Keras/TensorFlow.

Karena itu, kode berikut dibuat lebih aman dengan mengecek beberapa kemungkinan lokasi folder.

In [ ]:
base_dir = os.path.dirname(dataset)

candidate_dirs = [
    os.path.join(base_dir, "aclImdb"),
    os.path.join(base_dir, "aclImdb_v1_extracted", "aclImdb")
]

dataset_dir = None

for candidate in candidate_dirs:
    if os.path.exists(candidate):
        dataset_dir = candidate
        break

if dataset_dir is None:
    raise FileNotFoundError("Folder aclImdb tidak ditemukan. Cek kembali proses download dan ekstraksi dataset.")

print("Folder dataset:", dataset_dir)
print("Isi folder dataset:", os.listdir(dataset_dir))

## 9. Melihat Struktur Folder Training

Folder `train` berisi data latih. Di dalamnya ada folder `pos` dan `neg`.

Kita juga perlu memperhatikan folder `unsup`. Folder ini berisi data tanpa label, sehingga tidak digunakan untuk klasifikasi biner pada praktikum ini.

In [ ]:
train_dir = os.path.join(dataset_dir, "train")
test_dir = os.path.join(dataset_dir, "test")

print("Isi folder train:")
print(os.listdir(train_dir))

print("\nIsi folder test:")
print(os.listdir(test_dir))

## 10. Menghapus Folder `unsup`

Fungsi `text_dataset_from_directory` membaca subfolder sebagai kelas. Jika folder `unsup` tetap ada, TensorFlow dapat menganggapnya sebagai kelas tambahan.

Karena tugas kita hanya klasifikasi biner, kita hapus folder `unsup` jika folder tersebut ada.

Kode dibuat aman agar tidak error saat notebook dijalankan ulang.

In [ ]:
remove_dir = os.path.join(train_dir, "unsup")

if os.path.exists(remove_dir):
    shutil.rmtree(remove_dir)
    print("Folder 'unsup' berhasil dihapus.")
else:
    print("Folder 'unsup' tidak ada atau sudah pernah dihapus.")

## 11. Membuat Dataset Training dan Validation

Kita akan membuat dua dataset:

- `train_ds`: data untuk melatih model.
- `val_ds`: data untuk memantau performa model selama training.

Kita gunakan `validation_split=0.2`, artinya 20% data training dipakai sebagai validation set.

In [ ]:
batch_size = 1024
seed = 123

train_ds = tf.keras.utils.text_dataset_from_directory(
    train_dir,
    batch_size=batch_size,
    validation_split=0.2,
    subset="training",
    seed=seed
)

val_ds = tf.keras.utils.text_dataset_from_directory(
    train_dir,
    batch_size=batch_size,
    validation_split=0.2,
    subset="validation",
    seed=seed
)

## 12. Membuat Dataset Test

Selain training dan validation, kita juga dapat membuat `test_ds`.

Dataset test digunakan untuk evaluasi akhir setelah model selesai dilatih.

In [ ]:
test_ds = tf.keras.utils.text_dataset_from_directory(
    test_dir,
    batch_size=batch_size
)

## 13. Melihat Contoh Data

Sekarang kita lihat beberapa contoh ulasan dan labelnya.

Secara default, label biasanya:

- `0` = kelas pertama secara alfabetis.
- `1` = kelas kedua secara alfabetis.

Kita cek nama kelasnya terlebih dahulu.

In [ ]:
print("Nama kelas:", train_ds.class_names)

for text_batch, label_batch in train_ds.take(1):
    for i in range(3):
        print("=" * 80)
        print("Label:", label_batch[i].numpy())
        print("Teks:")
        print(text_batch.numpy()[i][:500])

## 14. Optimasi Pipeline Dataset

Saat training, model membaca data dari disk. Agar training lebih efisien, kita gunakan:

- `.cache()`: menyimpan data yang sudah dibaca agar tidak terus-menerus dibaca ulang.
- `.prefetch()`: menyiapkan batch berikutnya saat model sedang training batch saat ini.

Untuk pemula, cukup pahami bahwa dua baris ini membantu training berjalan lebih lancar.

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE

train_ds = train_ds.cache().prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.cache().prefetch(buffer_size=AUTOTUNE)
test_ds = test_ds.cache().prefetch(buffer_size=AUTOTUNE)

## 15. Text Preprocessing dengan `TextVectorization`

Sebelum masuk ke layer `Embedding`, teks harus diubah menjadi deretan integer.

Contoh:

```text
"film ini bagus"
```

menjadi:

```text
[45, 12, 789, 0, 0, 0, ...]
```

Layer `TextVectorization` akan melakukan beberapa hal:

1. Mengubah teks menjadi huruf kecil.
2. Menghapus tag HTML seperti `<br />`.
3. Menghapus tanda baca.
4. Memecah teks menjadi token/kata.
5. Mengubah token menjadi indeks angka.
6. Menyamakan panjang sequence dengan padding/truncation.

Kita batasi vocabulary menjadi 10.000 token paling sering, dan panjang setiap review menjadi 100 token.

In [ ]:
def custom_standardization(input_data):
    # Fungsi ini membersihkan teks sebelum diubah menjadi angka.
    #
    # Langkah:
    # 1. Ubah semua huruf menjadi lowercase.
    # 2. Hapus tag HTML <br />.
    # 3. Hapus tanda baca.
    lowercase = tf.strings.lower(input_data)
    stripped_html = tf.strings.regex_replace(lowercase, "<br />", " ")
    cleaned_text = tf.strings.regex_replace(
        stripped_html,
        "[%s]" % re.escape(string.punctuation),
        ""
    )
    return cleaned_text


vocab_size = 10000
sequence_length = 100

vectorize_layer = TextVectorization(
    standardize=custom_standardization,
    max_tokens=vocab_size,
    output_mode="int",
    output_sequence_length=sequence_length
)

# Untuk membangun vocabulary, TextVectorization hanya membutuhkan teks, bukan label.
text_ds = train_ds.map(lambda text, label: text)

# adapt() akan membaca data training dan membangun vocabulary berdasarkan token yang sering muncul.
vectorize_layer.adapt(text_ds)

print("Vocabulary berhasil dibuat.")
print("Jumlah token dalam vocabulary:", len(vectorize_layer.get_vocabulary()))

## 16. Melihat Vocabulary

Vocabulary adalah daftar kata/token yang dikenali oleh `TextVectorization`.

Beberapa token awal biasanya memiliki makna khusus:

- `''` dapat mewakili padding.
- `[UNK]` mewakili kata yang tidak dikenal atau tidak masuk vocabulary.

In [ ]:
vocab = vectorize_layer.get_vocabulary()

print("20 token pertama dalam vocabulary:")
for i, token in enumerate(vocab[:20]):
    print(i, repr(token))

## 17. Mencoba Vectorization pada Contoh Review

Kita ambil satu review dari dataset, lalu lihat bagaimana teks diubah menjadi integer.

In [ ]:
for text_batch, label_batch in train_ds.take(1):
    contoh_teks = text_batch[0]
    contoh_label = label_batch[0]
    break

hasil_vector = vectorize_layer(tf.expand_dims(contoh_teks, -1))

print("Label:", contoh_label.numpy())
print("\nTeks asli:")
print(contoh_teks.numpy()[:500])

print("\nHasil vectorization, 30 token pertama:")
print(hasil_vector.numpy()[0][:30])

## 18. Membuat Model Klasifikasi Sentimen

Arsitektur model kita sederhana:

```text
TextVectorization
        ↓
Embedding
        ↓
GlobalAveragePooling1D
        ↓
Dense(16, relu)
        ↓
Dense(1)
```

Penjelasan setiap layer:

1. **TextVectorization**  
   Mengubah teks mentah menjadi indeks angka.

2. **Embedding**  
   Mengubah indeks kata menjadi vektor embedding.

3. **GlobalAveragePooling1D**  
   Mengambil rata-rata embedding semua token dalam review, sehingga setiap review menjadi satu vektor tetap.

4. **Dense(16, relu)**  
   Layer neural network untuk mempelajari pola dari vektor review.

5. **Dense(1)**  
   Menghasilkan satu nilai logit untuk klasifikasi biner.

In [ ]:
embedding_dim = 16

model = Sequential([
    vectorize_layer,
    Embedding(
        input_dim=vocab_size,
        output_dim=embedding_dim,
        name="embedding"
    ),
    GlobalAveragePooling1D(),
    Dense(16, activation="relu"),
    Dense(1)
])

model.summary()

## 19. Compile Model

Sebelum training, model perlu dikompilasi.

Kita menggunakan:

- Optimizer: `adam`
- Loss: `BinaryCrossentropy(from_logits=True)`
- Metric: `accuracy`

Mengapa `from_logits=True`?

Karena layer terakhir `Dense(1)` tidak memakai fungsi aktivasi sigmoid. Jadi output model masih berupa logit, bukan probabilitas.

In [ ]:
model.compile(
    optimizer="adam",
    loss=tf.keras.losses.BinaryCrossentropy(from_logits=True),
    metrics=["accuracy"]
)

## 20. Melatih Model

Sekarang kita latih model.

Untuk praktikum kelas, `epochs=5` biasanya cukup agar waktu training tidak terlalu lama. Jika ingin hasil lebih baik, nilai ini dapat dinaikkan, misalnya menjadi 10 atau 15.

In [ ]:
# Callback TensorBoard akan menyimpan log training ke folder "logs".
# Log ini bisa dipakai untuk melihat grafik training secara interaktif.
tensorboard_callback = tf.keras.callbacks.TensorBoard(log_dir="logs")

epochs = 5

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=epochs,
    callbacks=[tensorboard_callback]
)

## 21. Visualisasi Akurasi dan Loss

Kita plot akurasi dan loss agar lebih mudah melihat proses belajar model.

Yang perlu diamati:

- Jika training accuracy naik, model belajar dari data training.
- Jika validation accuracy juga naik, model mampu melakukan generalisasi.
- Jika training accuracy jauh lebih tinggi daripada validation accuracy, kemungkinan terjadi overfitting.

In [ ]:
import matplotlib.pyplot as plt

history_dict = history.history

plt.figure()
plt.plot(history_dict["accuracy"], label="Training Accuracy")
plt.plot(history_dict["val_accuracy"], label="Validation Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Training vs Validation Accuracy")
plt.legend()
plt.show()

plt.figure()
plt.plot(history_dict["loss"], label="Training Loss")
plt.plot(history_dict["val_loss"], label="Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training vs Validation Loss")
plt.legend()
plt.show()

## 21B. Melihat Training dengan TensorBoard

TensorBoard adalah alat visualisasi dari TensorFlow. Pada praktikum ini, TensorBoard bersifat opsional.

Jika notebook dijalankan di Jupyter atau Colab, cell berikut dapat menampilkan dashboard training dari folder `logs`.

Jika dashboard tidak muncul di lingkungan lokal, bagian ini boleh dilewati karena kita sudah membuat grafik accuracy dan loss dengan Matplotlib.

In [ ]:
# Opsional: jalankan cell ini jika ingin membuka TensorBoard langsung dari notebook.
# Jika cell ini error di Jupyter lokal, bagian ini boleh dilewati.

%load_ext tensorboard
%tensorboard --logdir logs

## 22. Evaluasi pada Dataset Test

Setelah training selesai, kita evaluasi model pada data test.

Data test tidak digunakan saat training, sehingga hasilnya lebih mencerminkan kemampuan model pada data baru.

In [ ]:
test_loss, test_accuracy = model.evaluate(test_ds)

print("Test Loss    :", test_loss)
print("Test Accuracy:", test_accuracy)

## 23. Membuat Prediksi pada Teks Baru

Karena `TextVectorization` sudah menjadi bagian dari model, kita bisa langsung memasukkan teks mentah ke model.

Output model masih berupa logit. Untuk mengubahnya menjadi probabilitas, gunakan fungsi sigmoid:

```python
tf.sigmoid(logit)
```

Interpretasi sederhana:

- Probabilitas >= 0.5 → positif.
- Probabilitas < 0.5 → negatif.

In [ ]:
contoh_review = tf.constant([
    "This movie was fantastic, emotional, and beautifully acted.",
    "The story was boring and the acting was terrible.",
    "I did not expect much, but this film was surprisingly enjoyable."
])

logits = model.predict(contoh_review)
probabilities = tf.sigmoid(logits)

for review, prob in zip(contoh_review.numpy(), probabilities.numpy()):
    label_prediksi = "positif" if prob[0] >= 0.5 else "negatif"
    print("=" * 80)
    print("Review:", review.decode("utf-8"))
    print("Probabilitas positif:", float(prob[0]))
    print("Prediksi:", label_prediksi)

## 24. Mengambil Word Embeddings yang Sudah Dilatih

Embedding yang dipelajari model tersimpan sebagai bobot pada layer bernama `"embedding"`.

Bobot embedding berbentuk matriks:

```text
(vocab_size, embedding_dim)
```

Jika `vocab_size = 10000` dan `embedding_dim = 16`, maka matriks embedding berukuran:

```text
10000 x 16
```

Setiap baris mewakili satu token dalam vocabulary.

In [ ]:
weights = model.get_layer("embedding").get_weights()[0]
vocab = vectorize_layer.get_vocabulary()

print("Shape matriks embedding:", weights.shape)
print("Jumlah token vocabulary :", len(vocab))

## 25. Melihat Embedding untuk Beberapa Kata

Kita dapat mengambil embedding untuk kata tertentu dengan cara:

1. Cari indeks kata di vocabulary.
2. Ambil baris embedding sesuai indeks tersebut.

Catatan: Jika kata tidak ada di vocabulary, kata tersebut masuk kategori `[UNK]`.

In [ ]:
def tampilkan_embedding(kata, jumlah_dimensi=8):
    kata = kata.lower()

    if kata in vocab:
        index = vocab.index(kata)
    else:
        index = vocab.index("[UNK]")
        print(f"Kata '{kata}' tidak ada di vocabulary. Menggunakan token [UNK].")

    vector = weights[index]

    print("Kata:", kata)
    print("Index:", index)
    print(f"{jumlah_dimensi} nilai pertama embedding:")
    print(vector[:jumlah_dimensi])


tampilkan_embedding("beautiful")
tampilkan_embedding("terrible")

## 26. Mencari Kata yang Embedding-nya Mirip

Sekarang kita coba melihat kata-kata yang vektornya paling dekat dengan kata tertentu.

Kita gunakan **cosine similarity**.

Intuisi:

- Cosine similarity mendekati `1` berarti dua vektor sangat mirip arahnya.
- Cosine similarity mendekati `0` berarti tidak terlalu mirip.
- Cosine similarity mendekati `-1` berarti arahnya berlawanan.

Catatan penting untuk pemula:

> Karena model ini sederhana dan datasetnya terbatas, hasil kemiripan kata belum tentu sempurna secara semantik. Tujuannya adalah memahami proses, bukan menghasilkan embedding terbaik.

In [ ]:
import numpy as np

def cosine_similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-10)

def kata_terdekat(kata, top_k=10):
    kata = kata.lower()

    if kata in vocab:
        index = vocab.index(kata)
    else:
        index = vocab.index("[UNK]")
        print(f"Kata '{kata}' tidak ada di vocabulary. Menggunakan token [UNK].")

    target_vector = weights[index]

    similarities = []
    for i, token in enumerate(vocab):
        if i == 0:
            continue  # skip padding
        sim = cosine_similarity(target_vector, weights[i])
        similarities.append((token, sim))

    similarities = sorted(similarities, key=lambda x: x[1], reverse=True)

    print(f"Kata terdekat dengan '{kata}':")
    for token, sim in similarities[:top_k]:
        print(f"{token:20s} {sim:.4f}")

kata_terdekat("beautiful", top_k=10)

## 27. Menyimpan Embedding ke File `.tsv`

Kita akan menyimpan dua file:

1. `vectors.tsv`  
   Berisi angka vektor embedding.

2. `metadata.tsv`  
   Berisi token/kata yang sesuai dengan setiap baris vektor.

File ini dapat digunakan untuk visualisasi embedding, misalnya dengan **TensorBoard Embedding Projector**.

In [ ]:
out_v = io.open("vectors.tsv", "w", encoding="utf-8")
out_m = io.open("metadata.tsv", "w", encoding="utf-8")

for index, word in enumerate(vocab):
    if index == 0:
        continue  # skip token padding
    vec = weights[index]
    out_v.write("\t".join([str(x) for x in vec]) + "\n")
    out_m.write(word + "\n")

out_v.close()
out_m.close()

print("File berhasil dibuat:")
print("- vectors.tsv")
print("- metadata.tsv")

## 28. Download File Embedding Jika Menggunakan Google Colab

Jika notebook dijalankan di Google Colab, cell berikut dapat digunakan untuk mengunduh file `vectors.tsv` dan `metadata.tsv`.

Jika dijalankan di Jupyter lokal, cell ini akan dilewati secara otomatis.

In [ ]:
try:
    from google.colab import files
    files.download("vectors.tsv")
    files.download("metadata.tsv")
except Exception:
    print("Bukan environment Google Colab, atau fitur download Colab tidak tersedia.")

## 29. Visualisasi Embedding

Untuk memvisualisasikan embedding:

1. Buka **TensorBoard Embedding Projector**.
2. Pilih menu untuk memuat data.
3. Upload:
   - `vectors.tsv`
   - `metadata.tsv`
4. Cari kata tertentu, misalnya:
   - `beautiful`
   - `terrible`
   - `good`
   - `bad`

Dengan visualisasi, mahasiswa dapat melihat bagaimana kata-kata direpresentasikan sebagai titik dalam ruang vektor.

Namun perlu diingat:

> Model kita sederhana. Untuk embedding yang lebih stabil dan bermakna, biasanya dibutuhkan dataset lebih besar, arsitektur lebih kuat, atau algoritma khusus seperti Word2Vec.

## 30. Ringkasan Konsep

Pada notebook ini, kita telah mempelajari:

1. **Teks harus diubah menjadi angka** sebelum diproses oleh model machine learning.
2. **One-hot encoding** mudah dipahami tetapi boros dan sparse.
3. **Integer encoding** lebih ringkas tetapi tidak membawa makna hubungan antarkata.
4. **Word embedding** merepresentasikan kata sebagai vektor padat yang dapat dipelajari.
5. Layer **TextVectorization** mengubah teks mentah menjadi deretan indeks integer.
6. Layer **Embedding** mengubah indeks integer menjadi vektor embedding.
7. Model sederhana dapat belajar embedding sambil belajar melakukan klasifikasi sentimen.
8. Embedding yang sudah dilatih dapat disimpan dan divisualisasikan.

## Latihan

Coba eksperimen berikut:

1. Ubah `embedding_dim` dari `16` menjadi `8`, `32`, atau `64`.
2. Ubah `sequence_length` dari `100` menjadi `200`.
3. Ubah `epochs` dari `5` menjadi `10`.
4. Hapus layer `Dense(16, activation="relu")`, lalu bandingkan hasilnya.
5. Cari kata lain pada fungsi `kata_terdekat()`, misalnya `"boring"`, `"excellent"`, atau `"awful"`.

## Referensi

- TensorFlow Guide: Word embeddings  
  https://www.tensorflow.org/text/guide/word_embeddings
- Keras API: TextVectorization  
  https://keras.io/api/layers/preprocessing_layers/text/text_vectorization/
- Keras API: Embedding layer  
  https://keras.io/api/layers/core_layers/embedding/
- TensorBoard Embedding Projector  
  https://projector.tensorflow.org/